**Diplay the Product Table. The columns are separated with " "**

In [0]:
path = "/Volumes/lakehouse/adventureworks_multisales/raw_zone/Product.csv"
product_uncleaned = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", "\t")
    .load(path))
display(product_uncleaned)


**Checking the data types.**
-  We need to change the Standard Cost from string to int

In [0]:
product_uncleaned.printSchema()

**Inspection for nulls, distinct values and whitespace**


In [0]:
from pyspark.sql import functions as F

inspection = []
for c in product_uncleaned.columns:
    inspection.append((
        c,
        product_uncleaned.filter(F.col(c).isNull()).count(),
        product_uncleaned.select(c).distinct().count(),
        product_uncleaned.filter(F.col(c) != F.trim(F.col(c))).count()
    ))

spark.createDataFrame(inspection, ['column', 'null_count', 'distinct_count', 'trim_count']).show()



- ProductKey (Distinct: 397)
- Product (Distinct: 295)
- we have to drop duplicate product keys


In [0]:
total_rows = product_uncleaned.count()
unique_rows = product_uncleaned.distinct().count()

duplicate_count = total_rows - unique_rows
print("Total duplicated rows:", duplicate_count)

product_uncleaned.groupBy("ProductKey").count().filter("count > 1").show()

Inspection of duplicated rows, and checking which will be keeped


In [0]:
duplicated_keys = [388, 520, 223, 582, 455, 446]
(
product_uncleaned.filter(F.col("ProductKey").isin(duplicated_keys)) 
                 .orderBy("ProductKey") 
                 .show(truncate=False)

)

Inspection of inconsistent formating 
- Color : We have to change Multi and MULTI , Red and RED, Yellow and YELLOW 
- Subcategory : Wheels and Whels 
- Category : ok
- Backround Color Format :  ok
- Font Color Format: ok

In [0]:
product_uncleaned.select("Color").distinct().show()
product_uncleaned.select("Subcategory").distinct().show()
product_uncleaned.select("Category").distinct().show()
product_uncleaned.select("Background Color Format").distinct().show()
product_uncleaned.select("Font Color Format").distinct().show()



**Inspecting the Standard Cost column for empty strings, commas, symbols**
- There are no empty strings
- Except of commas and $ signs, we dont have letters, or other symbols to clean 


In [0]:
from pyspark.sql import functions as F
product_uncleaned.filter(F.col("Standard Cost") == "").count()

In [0]:
product_uncleaned.filter(~F.col("Standard Cost").rlike("^[0-9.,$]+$")).show(truncate=False)